> Same model like old one, but for three channel image

In [ ]:
#| default_exp three_channel_model

In [ ]:
#| export
import tensorflow as tf
from tensorflow.keras.layers import Layer
from tensorflow.keras.models import Sequential
from tensorflow.keras import backend as K

In [ ]:
#| export
from typing import List, Tuple, Union

In [ ]:
#| export
def pooling(
           inputs, 
           max_pool_only=False, 
           both=True, 
           pool_size=2):
    if both:
        p1 = tf.keras.layers.MaxPooling2D(
                                          (pool_size, pool_size))(inputs)
        p2 = tf.keras.layers.AvgPool2D(
                                        pool_size=(pool_size, pool_size))(inputs)
        return tf.keras.layers.concatenate([p1, p2])
    elif max_pool_only: return  tf.keras.layers.MaxPooling2D(
                                          (pool_size, pool_size))(inputs)
    else: return tf.keras.layers.AvgPool2D(
                                pool_size=(pool_size, pool_size)
                                )(inputs)

In [ ]:
#| export
def conv_block(
        inputs, 
        filter_no, 
        kernel_size, 
        batch_nm=True, 
        dropout=True, 
        drp_rt=0.1):
        
    c1 = tf.keras.layers.Conv2D(
                                filter_no, 
                                (kernel_size, kernel_size), 
                                kernel_initializer='he_normal',
                                padding='same',
                                activation=None)(inputs)
    if batch_nm:
        c1 = tf.keras.layers.BatchNormalization()(c1)
    c1 = tf.keras.layers.Activation('relu')(c1)
    if dropout:
        c1 = tf.keras.layers.Dropout(drp_rt)(c1)
    return c1       

In [ ]:
#| export
def double_conv_block(
        x, 
        kernel_size,
        filter_no, 
        dropout, 
        batch_norm=False):
    'conv + batchnorm + relu + conv + batchnorm + relu + dropout'
    
    conv = tf.keras.layers.Conv2D(
        filter_no, (kernel_size, kernel_size), padding="same")(x)
    if batch_norm is True:
        conv = tf.keras.layers.BatchNormalization(axis=3)(conv)
    conv = tf.keras.layers.Activation("relu")(conv)

    conv = tf.keras.layers.Conv2D(
        filter_no, 
        (kernel_size, kernel_size), 
        padding="same")(conv)
    if batch_norm is True:
        conv = tf.keras.layers.BatchNormalization(axis=3)(conv)
    conv = tf.keras.layers.Activation("relu")(conv)
    
    if dropout > 0:
        conv = tf.keras.layers.Dropout(dropout)(conv)

    return conv

In [ ]:
#| export
def res_conv_block(x, filter_size, size, dropout, batch_norm=False):
    '''
    conv -> batchno    conv -> batchnorm -> relu -> conv -> batchnorm -> dropout -> shortcut -> add -> relu
    Residual convolutional layer.
     https://arxiv.org/ftp/arxiv/papers/1802/1802.06955.pdf
    '''

    conv = tf.keras.layers.Conv2D(
        size, 
        (filter_size, filter_size), 
        padding='same')(x)
    if batch_norm is True:
        conv = tf.keras.layers.BatchNormalization(axis=3)(conv)
    conv = tf.keras.layers.Activation('relu')(conv)
    
    conv = tf.keras.layers.Conv2D(size, (filter_size, filter_size), padding='same')(conv)
    if batch_norm is True:
        conv = tf.keras.layers.BatchNormalization(axis=3)(conv)
    #conv = layers.Activation('relu')(conv)    #Activation before addition with shortcut
    if dropout > 0:
        conv = tf.keras.layers.Dropout(dropout)(conv)

    shortcut = tf.keras.layers.Conv2D(size, kernel_size=(1, 1), padding='same')(x)
    if batch_norm is True:
        shortcut = tf.keras.layers.BatchNormalization(axis=3)(shortcut)

    res_path = tf.keras.layers.add([shortcut, conv])
    res_path = tf.keras.layers.Activation('relu')(res_path)    #Activation after addition with shortcut (Original residual block)
    return res_path

In [ ]:
#| export
def gating_signal(input, out_size, batch_norm=False):
    """
    resize the down layer feature map into the same dimension as the up layer feature map
    using 1x1 conv
    :return: the gating feature map with the same dimension of the up layer feature map
    """
    x = tf.keras.layers.Conv2D(
        out_size, 
        (1, 1), 
        padding='same')(input)
    if batch_norm:
        x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Activation('relu')(x)
    return x

In [ ]:
#| export
def repeat_elem(tensor, rep):
    # lambda function to repeat Repeats the elements of a tensor along an axis
    #by a factor of rep.
    # If tensor has shape (None, 256,256,3), lambda will return a tensor of shape 
    #(None, 256,256,6), if specified axis=3 and rep=2.

     return tf.keras.layers.Lambda(
          lambda x, repnum: K.repeat_elements(x, repnum, axis=3),
                          arguments={'repnum': rep})(tensor)

In [ ]:
#| export
def attention_block(x, gating, inter_shape):
    shape_x = K.int_shape(x)
    shape_g = K.int_shape(gating)

# Getting the x signal to the same shape as the gating signal
    theta_x = tf.keras.layers.Conv2D(
        inter_shape, 
        (2, 2), 
        strides=(2, 2), 
        padding='same')(x)  # 16
    shape_theta_x = K.int_shape(theta_x)

# Getting the gating signal to the same number of filters as the inter_shape
    phi_g = tf.keras.layers.Conv2D(
        inter_shape, 
        (1, 1), 
        padding='same')(gating)
    upsample_g = tf.keras.layers.Conv2DTranspose(
        inter_shape, (3, 3),
        strides=(shape_theta_x[1] // shape_g[1], 
                 shape_theta_x[2] // shape_g[2]),
        padding='same')(phi_g)  # 16

    concat_xg = tf.keras.layers.add([upsample_g, theta_x])
    act_xg = tf.keras.layers.Activation('relu')(concat_xg)
    psi = tf.keras.layers.Conv2D(
        1, 
        (1, 1), 
        padding='same')(act_xg)
    sigmoid_xg = tf.keras.layers.Activation(
        'sigmoid')(psi)
    shape_sigmoid = K.int_shape(sigmoid_xg)
    upsample_psi = tf.keras.layers.UpSampling2D(
        size=(shape_x[1] // shape_sigmoid[1], shape_x[2] // shape_sigmoid[2]))(sigmoid_xg)  # 32

    upsample_psi = repeat_elem(upsample_psi, shape_x[3])

    y = tf.keras.layers.multiply([upsample_psi, x])

    result = tf.keras.layers.Conv2D(shape_x[3], (1, 1), padding='same')(y)
    result_bn = tf.keras.layers.BatchNormalization()(result)
    return result_bn

In [ ]:
#| export
def encode_128(
        input_size:Tuple[int, int]
    ):
    ' Encoder block for 128x128 input size'

    inputs = tf.keras.layers.Input(input_size)
    s = inputs
    c1 = conv_block(
                 inputs=s, 
                 filter_no=16, 
                 kernel_size=3, 
                 batch_nm=True, 
                 dropout=True, 
                 drp_rt=0.2)
     
    c1 = conv_block(
                 inputs=c1, 
                 filter_no=16, 
                 kernel_size=3, 
                 batch_nm=True, 
                 dropout=False, 
                 drp_rt=0.001) # NQA
    
    p1 = pooling(inputs=c1, both=True, pool_size=2)
    
    c2 = conv_block(
                   inputs=p1,
                   filter_no=32,
                   kernel_size=3,
                   batch_nm=True,
                   dropout=True,
                   drp_rt=0.2)
    
    c2 = conv_block(
                    inputs=c2,
                    filter_no=32,
                    kernel_size=3,
                    batch_nm=True,
                    dropout=False,
                    drp_rt=0.0001 # will not be used
                    )
    
    p2 = pooling(inputs=c2, both=True, pool_size=2)
    
    c3 = conv_block(
                   inputs=p2,
                   filter_no=64,
                   kernel_size=3,
        
                   dropout=True,
                   drp_rt=0.2)
    
    c3 = conv_block(
                   inputs=c3,
                   filter_no=64,
                   kernel_size=3,
                   dropout=False,
                   drp_rt=0.001 # will not be used
                   )
    return s, c1, c2, c3

In [ ]:
#| export
def encoder_block_256(
        input_size:Tuple[int, int, int]
    ):
    ' Encoder block for 256x256 input size'

    inputs = tf.keras.layers.Input(input_size)
    s = inputs
    c1 = conv_block(
                 inputs=s, 
                 filter_no=16, 
                 kernel_size=3, 
                 batch_nm=True, 
                 dropout=True, 
                 drp_rt=0.2)
     
    c1 = conv_block(
                 inputs=c1, 
                 filter_no=16, 
                 kernel_size=3, 
                 batch_nm=True, 
                 dropout=False, 
                 drp_rt=0.001) # NQA
    
    p1 = pooling(inputs=c1, both=True, pool_size=2)
    
    c2 = conv_block(
                   inputs=p1,
                   filter_no=32,
                   kernel_size=3,
                   batch_nm=True,
                   dropout=True,
                   drp_rt=0.2)
    
    c2 = conv_block(
                    inputs=c2,
                    filter_no=32,
                    kernel_size=3,
                    batch_nm=True,
                    dropout=False,
                    drp_rt=0.0001 # will not be used
                    )
    
    p2 = pooling(inputs=c2, both=True, pool_size=2)
    
    c3 = conv_block(
                   inputs=p2,
                   filter_no=64,
                   kernel_size=3,
        
                   dropout=True,
                   drp_rt=0.2)
    
    c3 = conv_block(
                   inputs=c3,
                   filter_no=64,
                   kernel_size=3,
                   dropout=False,
                   drp_rt=0.001 # will not be used
                   )

    p3 = pooling(inputs=c3, both=True, pool_size=2)
    c4 = conv_block(
                   inputs=p3,
                   filter_no=128,
                   kernel_size=3,
        
                   dropout=True,
                   drp_rt=0.2)
    
    c4 = conv_block(
                   inputs=c4,
                   filter_no=128,
                   kernel_size=3,
                   dropout=False,
                   drp_rt=0.001 # will not be used
                   )
    return s, c1, c2, c3, c4

In [ ]:
s, c1, c2, c3 = encode_128((128, 128, 1))
print(f'input shape = {s.shape}')
print(f'c1 shape = {c1.shape}')
print(f'c2 shape = {c2.shape}')    
print(f'c3 shape = {c3.shape}')

In [ ]:
s, c1, c2, c3, c4 = encoder_block_256((256, 256, 1))
print(f'input shape = {s.shape}')
print(f'c1 shape = {c1.shape}')
print(f'c2 shape = {c2.shape}')    
print(f'c3 shape = {c3.shape}')
print(f'c4 shape = {c4.shape}')

In [ ]:
#| export
def decoder_block_128(c1, c2, c3):
    
    # one concat block
    u4 = tf.keras.layers.Conv2DTranspose(
                                         32, 
                                         (2, 2), 
                                         strides=(2, 2), 
                                         padding='same')(c3)
    #if u4.shape != c2.shape:
        #c2_ =  c2[:, :, :152, :]
    #else:
    c2_ = c2
    u4 = tf.keras.layers.concatenate([u4, c2_])

    c4 = conv_block(
                   inputs=u4,
                   filter_no=32,
                   kernel_size=3,
                   batch_nm=True,
                   dropout=True,
                   drp_rt=0.2)

    c4 = conv_block(
                   inputs=c4,
                   filter_no=32,
                   kernel_size=3,
                   batch_nm=True,
                   dropout=False,
                   drp_rt=0.001, #will not be used
                   )

    # second concat block    
    u5 =   tf.keras.layers.Conv2DTranspose(
        16, 
        (2, 2), 
        strides=(2, 2), padding='same')(c4)   
    if u5.shape !=c1.shape:
        paddings = tf.constant([[0, 0], [0, 0], [0, 2], [0, 0]])
        u5_padded = tf.pad(u5, paddings)
    else:
        u5_padded=u5

    

    u5 = tf.keras.layers.concatenate([u5_padded, c1])
    c5 = conv_block(
                   inputs=u5,
                   filter_no=16,
                   kernel_size=3,
                   batch_nm=True,
                   dropout=True,
                   drp_rt=0.2)
    c5 = conv_block(
                   inputs=c5,
                   filter_no=16,
                   kernel_size=3,
                   batch_nm=True,
                   dropout=False,
                   drp_rt=0)
    return c5

In [ ]:
#| export
def decoder_block_256(c1, c2, c3, c4):
    
    # one concat block
    u5 = tf.keras.layers.Conv2DTranspose(
                                         64, 
                                         (2, 2), 
                                         strides=(2, 2), 
                                         padding='same')(c4)
    u5 = tf.keras.layers.concatenate([u5, c3])

    c5 = conv_block(
                   inputs=u5,
                   filter_no=64,
                   kernel_size=3,
                   batch_nm=True,
                   dropout=True,
                   drp_rt=0.2)

    c5 = conv_block(
                   inputs=c5,
                   filter_no=64,
                   kernel_size=3,
                   batch_nm=True,
                   dropout=False,
                   drp_rt=0.001, #will not be used
                   )

    # second concat block    
    u6 =   tf.keras.layers.Conv2DTranspose(
        32, 
        (2, 2), 
        strides=(2, 2), padding='same')(c5)   
    if u6.shape !=c2.shape:
        paddings = tf.constant([[0, 0], [0, 0], [0, 2], [0, 0]])
        u6_padded = tf.pad(u6, paddings)
    else:
        u6_padded=u6

    

    u6 = tf.keras.layers.concatenate([u6_padded, c2])
    c6 = conv_block(
                   inputs=u6,
                   filter_no=32,
                   kernel_size=3,
                   batch_nm=True,
                   dropout=True,
                   drp_rt=0.2)
    c6 = conv_block(
                   inputs=c6,
                   filter_no=32,
                   kernel_size=3,
                   batch_nm=True,
                   dropout=False,
                   drp_rt=0)

    u7 =  tf.keras.layers.Conv2DTranspose(
        16, 
        (2, 2), 
        strides=(2, 2), padding='same')(c6)

    u7 = tf.keras.layers.concatenate([u7, c1])

    c7 = conv_block(
                     inputs=u7,
                     filter_no=16,
                     kernel_size=3,
                     batch_nm=True,
                     dropout=True,
                     drp_rt=0.2)
    c7 = conv_block(    
                     inputs=c7,
                     filter_no=16,
                     kernel_size=3,
                     batch_nm=True,
                     dropout=False,
                     drp_rt=0)
    return c7

In [ ]:
c7 = decoder_block_256(c1, c2, c3, c4)
c7.shape

In [ ]:
#| export
def encoder_decoder_256(
        input_size:Tuple[int, int, int],
        n_classes:int
    ):
    ' Combine encoder and decoder blocks for 256x256 input size'
    s, c1, c2, c3, c4 = encoder_block_256(
        input_size)
    c7 = decoder_block_256(c1, c2, c3, c4)

    outputs = tf.keras.layers.Conv2D(
        n_classes, 
        (1, 1), 
        activation='sigmoid')(c7)
    return tf.keras.Model(
        inputs=s, 
        outputs=outputs)

In [ ]:
model = encoder_decoder_256((256, 256, 1), 1)

In [ ]:
model.summary()

In [ ]:
#| export
def encode_128_decoder_128(
        input_size:Tuple[int, int],
        n_classes:int
    ):
    s, c1, c2, c3 = encode_128(input_size)
    c5 = decoder_block_128(c1, c2, c3)

    outputs = tf.keras.layers.Conv2D(
        n_classes, 
        (1, 1), 
        activation='sigmoid')(c5)

    return tf.keras.Model(inputs=s, outputs=outputs)

In [ ]:
#| hide
IMAGE_HEIGHT = 128
IMAGE_WIDTH = 128
NUM_CHANNELS = 1
NUM_CLASSES = 1
input_size=(IMAGE_HEIGHT, IMAGE_WIDTH, NUM_CHANNELS)
model = encode_128_decoder_128(
    input_size=(IMAGE_HEIGHT, IMAGE_WIDTH, NUM_CHANNELS), 
    n_classes=NUM_CLASSES)


In [ ]:
model.summary()

In [ ]:
#| export
def encoder_block(input_size):
    
    inputs = tf.keras.layers.Input(input_size)
    s = inputs
    c1 = conv_block(
                 inputs=s, 
                 filter_no=16, 
                 kernel_size=3, 
                 batch_nm=True, 
                 dropout=True, 
                 drp_rt=0.2)
     
    c1 = conv_block(
                 inputs=c1, 
                 filter_no=16, 
                 kernel_size=3, 
                 batch_nm=True, 
                 dropout=False, 
                 drp_rt=0.001) # NQA
    
    p1 = pooling(inputs=c1, both=True, pool_size=2)
    
    c2 = conv_block(
                   inputs=p1,
                   filter_no=32,
                   kernel_size=3,
                   batch_nm=True,
                   dropout=True,
                   drp_rt=0.2)
    
    c2 = conv_block(
                    inputs=c2,
                    filter_no=32,
                    kernel_size=3,
                    batch_nm=True,
                    dropout=False,
                    drp_rt=0.0001 # will not be used
                    )
    
    p2 = pooling(inputs=c2, both=True, pool_size=2)
    
    c3 = conv_block(
                   inputs=p2,
                   filter_no=64,
                   kernel_size=3,
        
                   dropout=True,
                   drp_rt=0.2)
    
    c3 = conv_block(
                   inputs=c3,
                   filter_no=64,
                   kernel_size=3,
                   dropout=False,
                   drp_rt=0.001 # will not be used
                   )
    p3 = pooling(inputs=c3, both=True, pool_size=2)
    
    c4 = conv_block(
                   inputs=p3,
                   filter_no=128,
                   batch_nm=True,
                   kernel_size=3,
                   dropout=True,
                   drp_rt=0.2
                   )
    c4 = conv_block(
                   inputs=c4,
                   filter_no=128,
                   kernel_size=3,
                   batch_nm=True,
                   dropout=False,
                   drp_rt=0.001# will not be used
                   )
                  
        
    p4 = pooling(both=True, inputs=c4, pool_size=2)
    
    c5 = conv_block(
                   inputs=p4,
                   filter_no=256,
                   kernel_size=3,
                   batch_nm=True,
                   dropout=True,
                   drp_rt=0.3)
    c5 = conv_block(
                   inputs=c5,
                   filter_no=256,
                   kernel_size=3,
                   batch_nm=True,
                   dropout=False,
                   drp_rt=0.01 # will not be used
                   )
    return s, c1, c2, c3, c4, c5

In [ ]:
#| export
def decoder_block(c1, c2, c3, c4, c5):
    
    # one concat block
    u6 = tf.keras.layers.Conv2DTranspose(128, (2, 2), strides=(2, 2), padding='same')(c5)
    if u6.shape != c4.shape:
        c4_ =  c4[:, :, :152, :]
    else:
        c4_ = c4
    u6 = tf.keras.layers.concatenate([u6, c4_])

    c6 = conv_block(
                   inputs=u6,
                   filter_no=128,
                   kernel_size=3,
                   batch_nm=True,
                   dropout=True,
                   drp_rt=0.2)

    c6 = conv_block(
                   inputs=c6,
                   filter_no=128,
                   kernel_size=3,
                   batch_nm=True,
                   dropout=False,
                   drp_rt=0.001, #will not be used
                   )

    # second concat block    
    u7 =   tf.keras.layers.Conv2DTranspose(64, (2, 2), strides=(2, 2), padding='same')(c6)   
    if u7.shape !=c3.shape:
        paddings = tf.constant([[0, 0], [0, 0], [0, 2], [0, 0]])
        u7_padded = tf.pad(u7, paddings)
    else:
        u7_padded=u7

    u7 = tf.keras.layers.concatenate([u7_padded, c3])
    c7 = conv_block(
                   inputs=u7,
                   filter_no=64,
                   kernel_size=3,
                   batch_nm=True,
                   dropout=True,
                   drp_rt=0.2)
    c7 = conv_block(
                   inputs=c7,
                   filter_no=64,
                   kernel_size=3,
                   batch_nm=True,
                   dropout=False,
                   drp_rt=0)

    # third conccat layer
    u8 = tf.keras.layers.Conv2DTranspose(32, (2, 2), strides=(2, 2), padding='same')(c7)
    u8 = tf.keras.layers.concatenate([u8, c2])
    c8 = conv_block(
                   inputs=u8,
                   filter_no=32,
                   kernel_size=3,
                   batch_nm=True,
                   dropout=True,
                   drp_rt=0.2)
    c8 = conv_block(
                   inputs=c8,
                   filter_no=32,
                   kernel_size=3,
                   batch_nm=True,
                   dropout=False,
                   drp_rt=0)
    # 4th concat
    u9 = tf.keras.layers.Conv2DTranspose(16, (2,2), strides=(2, 2), padding='same')(c8)
    u9 = tf.keras.layers.concatenate([u9, c1])
    c9= conv_block(
                   inputs=u9,
                   filter_no=16,
                   kernel_size=3,
                   batch_nm=True,
                   dropout=True,
                   drp_rt=0.2)
    c9= conv_block(
               inputs=c9,
               filter_no=16,
               kernel_size=3,
               batch_nm=True,
               dropout=False,
               drp_rt=0)
    return c9

In [ ]:
#| export
def encoder_decoder(
                     input_size, 
                     n_classes=2):

    s, c1, c2, c3, c4, c5 = encoder_block(input_size)
    
    c9 = decoder_block(c1, c2, c3, c4, c5)
    
    outputs= tf.keras.layers.Conv2D(n_classes, (1, 1), activation='sigmoid')(c9)
    return tf.keras.models.Model(inputs=s, outputs=outputs)

In [ ]:
#| hide
IMAGE_HEIGHT = 1152
IMAGE_WIDTH = 1632
NUM_CHANNELS = 3
NUM_CLASSES = 1
input_size=(IMAGE_HEIGHT, IMAGE_WIDTH, NUM_CHANNELS)
model = encoder_decoder(input_size, NUM_CLASSES)


In [ ]:
#| hide
IMAGE_HEIGHT = 1152 // 4
IMAGE_WIDTH = 1632 // 4
NUM_CHANNELS = 3
NUM_CLASSES = 1
input_size=(IMAGE_HEIGHT, IMAGE_WIDTH, NUM_CHANNELS)
model = encoder_decoder(input_size, NUM_CLASSES)


In [ ]:
class SplitAndBatchLayer(Layer):
    def __init__(self, model,**kwargs):
        super(SplitAndBatchLayer, self).__init__(**kwargs)
        self.model = model

    def build(self, input_shape):
        height, width, channels = input_shape[1:]
        assert height == 1152 and width == 1632, "Input image must have dimensions 1152x1632"

        # Split the input image into 4 pieces
        self.piece_height = height // 2
        self.piece_width = width // 2

    def call(self, inputs):
        # Split the input image into 4 pieces
        pieces = [
            inputs[:, :self.piece_height, :self.piece_width, :],
            inputs[:, :self.piece_height, self.piece_width:, :],
            inputs[:, self.piece_height:, :self.piece_width, :],
            inputs[:, self.piece_height:, self.piece_width:, :]
        ]
        # Batch the 4 pieces into a single batch
        batched_pieces = tf.concat(pieces, axis=0)
           # Process the batched pieces with your model
        batched_predictions = self.model(batched_pieces)
         # Split the batched predictions back into 4 pieces
        predictions = tf.split(batched_predictions, 4, axis=0)

        # Combine the 4 pieces into a single output image
        output_height = self.piece_height * 2
        output_width = self.piece_width * 2

        output_image = tf.concat([
            tf.concat([predictions[0], predictions[1]], axis=2),
            tf.concat([predictions[2], predictions[3]], axis=2)
        ], axis=1)

        return output_image

    def compute_output_shape(self, input_shape):
        return (input_shape[0], 1152, 1632, input_shape[3])



In [ ]:
model = Sequential([
    tf.keras.layers.Conv2D(
        32, 
        (3, 3), 
        activation='relu', 
        input_shape=(IMAGE_HEIGHT, IMAGE_WIDTH, 3)),
    # Add more layers...
])

In [ ]:
split_and_batch_layer = SplitAndBatchLayer(model=model)

In [ ]:
model.summary()

In [ ]:
h,w,c = 1152, 1632, 3
p_h = h // 2
p_w = w // 2

In [ ]:
p_w *2

In [ ]:
p_h *2

In [ ]:
inputs = tf.keras.Input(shape=( 1152, 1632, 3))
outputs = split_and_batch_layer(inputs)

In [ ]:
#| hide
model.summary()

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export('09_three_channel_model.ipynb')